# 05 — End-to-End Pipeline Demo

Demonstrate the full two-stage cascade pipeline:

```
Image/Video → Stage 1 Detector → Crops
                                    ├─ Pothole → PotholeAnalyzer → severity + distance + instructions
                                    └─ Traffic → TrafficAnalyzer → color + lane_relevant + instructions
```

This notebook shows:
1. Single image inference with full analysis breakdown
2. Batch of images with annotated output
3. CV score visualisation per pothole
4. Traffic light color detection breakdown
5. Video inference (optional)

In [ ]:
import sys
sys.path.insert(0, '../src')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

BASE_DIR = Path('..').resolve()

from pipeline import Pipeline

pipe = Pipeline()
print('Pipeline ready.')

## 1. Single Image — Full Analysis Breakdown

In [ ]:
# Use a validation image — change path to your own dashcam image if available
val_dir = BASE_DIR / 'data' / 'processed' / 'detector_yolo' / 'images' / 'val'
sample_paths = sorted(val_dir.glob('*'))[:1]

if not sample_paths:
    print('No val images found — run prepare_data.py first')
else:
    img_path = sample_paths[0]
    frame    = cv2.imread(str(img_path))
    result   = pipe.run(frame)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(cv2.cvtColor(frame,                   cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original', fontsize=12)
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(result['annotated_frame'], cv2.COLOR_BGR2RGB))
    axes[1].set_title('Annotated (Stage 1 + Stage 2)', fontsize=12)
    axes[1].axis('off')
    plt.suptitle(img_path.name, fontsize=11)
    plt.tight_layout()
    plt.show()

    print('\n=== WARNINGS ===')
    for w in result['warnings'] or ['No warnings.']:
        print(' ', w)

## 2. Pothole Detection — Detailed Breakdown

In [ ]:
if result['potholes']:
    print(f'Detected {len(result["potholes"])} pothole(s):\n')
    for i, ph in enumerate(result['potholes']):
        print(f'  Pothole #{i+1}')
        print(f'    Confidence  : {ph["conf"]:.2%}')
        print(f'    Severity    : {ph["severity"]}')
        print(f'    Distance    : {ph["distance"]}')
        print(f'    Area (px)   : {ph["area_px"]}')
        print(f'    Instructions: {ph["instructions"]}')
        if ph.get('cv_scores'):
            s = ph['cv_scores']
            print(f'    CV scores   : edge={s["edge_density"]:.3f}  '
                  f'depth={s["depth_score"]:.3f}  '
                  f'texture={s["texture_score"]:.3f}  '
                  f'combined={s["combined"]:.3f}')
        print()
else:
    print('No potholes detected in this frame.')

## 3. CV Score Radar Chart (per pothole)

In [ ]:
potholes_with_scores = [p for p in result['potholes'] if p.get('cv_scores')]

if potholes_with_scores:
    metrics_keys   = ['edge_density', 'depth_score', 'texture_score']
    metrics_labels = ['Edge Density', 'Depth Score', 'Texture Score']
    n_metrics = len(metrics_keys)
    angles    = np.linspace(0, 2*np.pi, n_metrics, endpoint=False).tolist()
    angles   += angles[:1]

    fig, axes = plt.subplots(1, len(potholes_with_scores),
                              figsize=(5*len(potholes_with_scores), 5),
                              subplot_kw={'polar': True})
    if len(potholes_with_scores) == 1: axes = [axes]

    sev_colors = {'Low': '#27ae60', 'Medium': '#f39c12', 'High': '#e74c3c'}

    for ax, ph in zip(axes, potholes_with_scores):
        values = [ph['cv_scores'][k] for k in metrics_keys]
        values_norm = [min(v / 0.3, 1.0) for v in values]  # normalise to 0-1
        values_norm += values_norm[:1]

        color = sev_colors.get(ph['severity'], 'gray')
        ax.plot(angles, values_norm, color=color, linewidth=2)
        ax.fill(angles, values_norm, color=color, alpha=0.25)
        ax.set_thetagrids(np.degrees(angles[:-1]), metrics_labels)
        ax.set_ylim(0, 1)
        ax.set_title(f'Severity: {ph["severity"]}\nConf: {ph["conf"]:.0%}',
                     color=color, fontsize=10, pad=15)

    plt.suptitle('Pothole CV Score Radar', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('No pothole CV scores to display.')

## 4. Traffic Light Detection — Detailed Breakdown

In [ ]:
if result['traffic_lights']:
    print(f'Detected {len(result["traffic_lights"])} traffic light(s):\n')
    for i, tl in enumerate(result['traffic_lights']):
        lane = 'YES (your lane)' if tl['lane_relevant'] else 'NO (adjacent)'
        print(f'  Traffic Light #{i+1}')
        print(f'    Confidence       : {tl["conf"]:.2%}')
        print(f'    Color            : {tl["color"]}')
        print(f'    Lane Relevant    : {lane}')
        print(f'    Color Confidence : {tl.get("color_confidence", "n/a")}')
        print(f'    Position Vote    : {tl.get("position_vote", "n/a")}')
        print(f'    Color Vote       : {tl.get("color_vote", "n/a")}')
        print(f'    Instructions     : {tl["instructions"]}')
        print()
else:
    print('No traffic lights detected in this frame.')

## 5. Batch Run — Val Images Grid

In [ ]:
val_images = sorted(val_dir.glob('*'))[:9]
fig, axes  = plt.subplots(3, 3, figsize=(15, 10))

for ax, img_path in zip(axes.flatten(), val_images):
    frame  = cv2.imread(str(img_path))
    result = pipe.run(frame)
    ann    = cv2.cvtColor(result['annotated_frame'], cv2.COLOR_BGR2RGB)
    ax.imshow(ann)
    ax.axis('off')
    # Summarise results in title
    ph_n  = len(result['potholes'])
    tl_n  = sum(1 for t in result['traffic_lights'] if t['lane_relevant'])
    title = f'PH:{ph_n} TL:{tl_n}'
    if result['potholes']:
        title += f' [{result["potholes"][0]["severity"]}]'
    ax.set_title(title, fontsize=8)

plt.suptitle('Batch Inference — Validation Set (Stage 1 + Stage 2)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Video Inference (Optional)

Set `VIDEO_PATH` to a dashcam video file to process it.

In [ ]:
VIDEO_PATH  = None   # e.g. '../sample_dashcam.mp4'
OUTPUT_PATH = '../output_annotated.mp4'

if VIDEO_PATH and Path(VIDEO_PATH).exists():
    result = pipe.run_video(VIDEO_PATH, OUTPUT_PATH)
    print(f'Processed {result["frame_count"]} frames → {OUTPUT_PATH}')
else:
    print('Set VIDEO_PATH to a dashcam video file to run video inference.')

## 7. Traffic Light Color Detection Internals

In [ ]:
from analyzers.traffic_analyzer import _detect_color

# Pick traffic light crops from val set to visualise the detection internals
tl_crops = []
for img_path in sorted(val_dir.glob('*'))[:50]:
    frame  = cv2.imread(str(img_path))
    if frame is None: continue
    preds  = pipe.detector(frame, conf=0.40, verbose=False)
    for box in preds[0].boxes:
        if int(box.cls[0]) == 1:  # traffic_light
            x1,y1,x2,y2 = [int(v) for v in box.xyxy[0]]
            crop = frame[y1:y2, x1:x2]
            if crop.size > 0:
                tl_crops.append(crop)

if tl_crops:
    n = min(len(tl_crops), 5)
    fig, axes = plt.subplots(2, n, figsize=(3*n, 6))

    for col, crop in enumerate(tl_crops[:n]):
        color, pos_vote, color_vote, conf = _detect_color(crop)

        # Top row: original crop
        axes[0][col].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        axes[0][col].axis('off')
        axes[0][col].set_title(f'Crop', fontsize=8)

        # Bottom row: show 48x144 resized with thirds marked
        resized = cv2.resize(crop, (48, 144))
        axes[1][col].imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB), aspect='auto')
        axes[1][col].axhline(48,  color='white', linewidth=1.5, linestyle='--')
        axes[1][col].axhline(96,  color='white', linewidth=1.5, linestyle='--')
        axes[1][col].set_xticks([]); axes[1][col].set_yticks([])
        c_map  = {'Red': '#e74c3c', 'Yellow': '#f1c40f', 'Green': '#2ecc71', 'Unknown': 'gray'}
        t_color = c_map.get(color, 'white')
        axes[1][col].set_title(
            f'Color: {color}\nPos:{pos_vote} HSV:{color_vote}\nConf:{conf:.1f}',
            fontsize=7, color=t_color
        )

    plt.suptitle('Traffic Light Color Detection Internals\n(dashed lines = spatial thirds)', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('No traffic light crops found in validation set.')